### Templating Alignment Data

In [7]:
from datasets import load_dataset

def format_prompt(example):
    """Format the prompt to using the <|user|> template TinyLlama is using"""
    # Format answers
    system = "<|system|>\n" + example["system"] + "</s>\n"
    prompt = "<|user|>\n" + example["input"] + "</s>\n<|assistant|>\n"
    chosen = example["chosen"] + "</s>\n"
    rejected = example["rejected"] + "</s>\n"
    return {
        "prompt": system + prompt,
        "chosen": chosen,
        "rejected": rejected
    }

# Apply formatting to the dataset and select relatively short answers
dpo_dataset = load_dataset(
    "argilla/distilabel-intel-orca-dpo-pairs", split="train"
    )

dpo_dataset = dpo_dataset.filter(
    lambda x: x["status"] != 'tie' and
    x['chosen_score'] >= 8 and
    not x['in_gsm8k_train']
)

dpo_dataset = dpo_dataset.map(format_prompt, remove_columns=dpo_dataset.column_names)
dpo_dataset #.save_to_disk("formatted_dpo_dataset")

Map:   0%|          | 0/5922 [00:00<?, ? examples/s]

Dataset({
    features: ['chosen', 'rejected', 'prompt'],
    num_rows: 5922
})

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import AutoPeftModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# 4-bit quantization configuration - Q in LORA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load the model with quantization on GPU
model = AutoPeftModelForCausalLM.from_pretrained(
    "tinyllama-finetuned", #"TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    quantization_config=bnb_config,
    device_map="auto"
)

merged_model = model.merge_and_unload()


# model.config.use_cache = False  # Disable caching for training
# model.config.pretraining_tp = 1  # Set tensor parallelism to 1 for training

# Load the Llama tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"  # Set the pad token
tokenizer.padding_side = "left"  # Pad on the left side

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the model for k-bit training
peft_config = LoraConfig(
    r=64,  # [4, 64] Rank of the low-rank decomposition. increasing this will increase the number of trainable parameters and potentially improve performance, but also increase memory usage and training time.
    lora_alpha=32,  # Scaling factor for the LORA updates. It balances the contribution of the LORA updates with the original model weights. Increasing this will increase the impact of the LORA updates on the model's weights, which can lead to faster convergence but may also cause instability if set too high. choose a value twice the value of r as a good starting point.
    lora_dropout=0.1,  # Dropout rate for LORA layers.
    bias="none",  # Whether to include bias in LORA layers
    task_type="CAUSAL_LM",  # Task type for LORA
    # init_lora_weights="gaussian",  # Initialization method for LORA weights
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]  # Target modules for LORA
)

# Prepare the model for k-bit training
model = prepare_model_for_kbit_training(model)
# Wrap the model with LORA
model = get_peft_model(model, peft_config)

In [10]:
from trl import DPOConfig

output_dir = "./tinyllama-dpo-finetuned"  # Directory to save the fine-tuned model and training outputs
training_args = DPOConfig(
    output_dir=output_dir,
    # num_train_epochs=1,  # Number of training epochs
    per_device_train_batch_size=4,  # Batch size per device during training
    gradient_accumulation_steps=4,  # Number of steps to accumulate gradients before updating model weights
    optim="paged_adamw_32bit",  # Optimizer to use (paged AdamW with 8-bit precision)
    learning_rate=1e-5,  # Learning rate for training
    lr_scheduler_type="cosine",  # Learning rate scheduler type
    max_steps=200,  # Maximum number of training steps
    logging_steps=10,  # Log training progress every 10 steps
    fp16=True,  # Use mixed precision training (fp16)
    gradient_checkpointing=True,  # Enable gradient checkpointing to save memory
    warmup_ratio=0.1,  # Warmup ratio for learning rate scheduler
)

In [11]:
from trl import DPOTrainer

# Create DPO Trainer
dpo_trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    peft_config=peft_config,
    beta=0.1,  # DPO beta parameter
    max_prompt_length=512,  # Maximum length of the prompt
    max_length=512,  # Maximum length of the generated response
)

# Finetune the model using DPO
dpo_trainer.train()

dpo_trainer.model.save_pretrained("TinyLlama-1.1B-dpo-qlora")

/home/monster/Desktop/NLP/llm/llm_venv/lib/python3.12/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_prompt_length, max_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in DPOTrainer, please use the DPOConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/home/monster/Desktop/NLP/llm/llm_venv/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:325: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/home/monster/Desktop/NLP/llm/llm_venv/lib/python3.12/site-packages/trl/trainer/dpo_trainer.py:358: UserWarning: You passed `max_length` to the DPOTrainer, the value you passed will override the one in the `DPOConfig`.
  warnings.warn(
/home/monster/Desktop/NLP/llm/llm_venv/lib/python3.12/site-packages/trl/trainer/dpo_trainer.py:371: UserWarning: You passed `max_prompt_length` to the DP

Map:   0%|          | 0/5922 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


  0%|          | 0/200 [00:00<?, ?it/s]

/home/monster/Desktop/NLP/llm/llm_venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/monster/Desktop/NLP/llm/llm_venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:91: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


{'loss': 0.6906, 'grad_norm': 2.4031333923339844, 'learning_rate': 5e-06, 'rewards/chosen': 0.00012565731594804674, 'rewards/rejected': -0.004999241791665554, 'rewards/accuracies': 0.3062500059604645, 'rewards/margins': 0.0051248990930616856, 'logps/rejected': -136.13644409179688, 'logps/chosen': -116.41001892089844, 'logits/rejected': -2.7850239276885986, 'logits/chosen': -2.6964316368103027, 'epoch': 0.03}
{'loss': 0.6691, 'grad_norm': 2.310025215148926, 'learning_rate': 9.5e-06, 'rewards/chosen': 0.0005475628422573209, 'rewards/rejected': -0.05022625997662544, 'rewards/accuracies': 0.5, 'rewards/margins': 0.050773829221725464, 'logps/rejected': -154.5161590576172, 'logps/chosen': -118.5445556640625, 'logits/rejected': -2.8061790466308594, 'logits/chosen': -2.6944124698638916, 'epoch': 0.05}
{'loss': 0.6501, 'grad_norm': 2.3929443359375, 'learning_rate': 9.938441702975689e-06, 'rewards/chosen': -0.031668733805418015, 'rewards/rejected': -0.1309693604707718, 'rewards/accuracies': 0.41

In [13]:
from peft import PeftModel

# Merge the LORA weights into the base model and unload the LORA adapter
model = AutoPeftModelForCausalLM.from_pretrained(
    "tinyllama-finetuned", #"TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto"
    )

sft_model = model.merge_and_unload()

# Merge DPO LoRA and SFT model
dpo_model = PeftModel.from_pretrained(
    sft_model,
    "TinyLlama-1.1B-dpo-qlora",
    device_map="auto"
)
dpo_model = dpo_model.merge_and_unload()

In [22]:
from transformers import pipeline

# Use the predefined prompt template
prompt = """<|user|>
Tell me something about diffusion generative models.</s>
<|assistant|>
"""

# Run our instruction-tuned model
pipe = pipeline(
    "text-generation",
    model=dpo_model,
    tokenizer=tokenizer,
    device_map="auto",
)
print(pipe(prompt, max_length=512, do_sample=True, temperature=0.7)[0]['generated_text'])

<|user|>
Tell me something about diffusion generative models.</s>
<|assistant|>
Diffusion generative models are a type of generative model that are capable of generating new data from a predefined distribution. They are commonly used in applications such as image generation, audio generation, and text generation. The model generates new data using a probabilistic distribution that is learned from training data.

One of the main advantages of diffusion generative models is their ability to generate high-quality data from a small amount of training data. This is because the model learns from the training data using a probabilistic framework, which allows it to generate new data that is similar to the input data.

Diffusion generative models can also be used to generate data that is not available in the training data. This is possible because the model can use the training data to generate new data while taking into account the distribution of the training data. This makes it possible for